In [4]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "X_train_features.npy" in files:
        print("FOUND:", os.path.join(root, "X_train_features.npy"))

FOUND: /content/drive/MyDrive/processed/processed/X_train_features.npy


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import numpy as np

DATA_PATH = "/content/drive/MyDrive/processed/processed"

X_train_raw = np.load(f"{DATA_PATH}/X_train_features.npy")
train_chan_ids = np.load(f"{DATA_PATH}/train_chan_ids.npy", allow_pickle=True)
X_test = np.load(f"{DATA_PATH}/X_features.npy")
y_test = np.load(f"{DATA_PATH}/y_labels.npy")
test_chan_ids = np.load(f"{DATA_PATH}/chan_ids.npy", allow_pickle=True)

def clean_features(X, chan_ids, y=None, name=""):
    bad_mask = np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1)
    print(f"{name}: {bad_mask.sum()} bad rows out of {len(X)}")
    X_clean = X[~bad_mask]; chan_ids_clean = chan_ids[~bad_mask]
    if y is not None:
        return X_clean, chan_ids_clean, y[~bad_mask]
    return X_clean, chan_ids_clean

X_train_raw, train_chan_ids = clean_features(X_train_raw, train_chan_ids, name="train/")
X_test, test_chan_ids, y_test = clean_features(X_test, test_chan_ids, y_test, name="test/")

print("train/:", X_train_raw.shape, "| test/:", X_test.shape, "| anomalous:", y_test.sum())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train/: 0 bad rows out of 9599
test/: 0 bad rows out of 25522
train/: (9599, 504) | test/: (25522, 504) | anomalous: 3459


In [15]:
import torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

SEQ_LEN = 8

class LSTMAutoencoder(nn.Module):
    def __init__(self, d, hidden=32, latent=8):
        super().__init__()
        self.encoder = nn.LSTM(d, hidden, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, d, batch_first=True)
    def encode(self, x):
        _, (h, _) = self.encoder(x)
        return self.to_latent(h[-1])
    def forward(self, x):
        z = self.encode(x)
        h_dec = self.from_latent(z).unsqueeze(1).repeat(1, x.size(1), 1)
        out, _ = self.decoder(h_dec)
        return out, z

def make_sequences(feats, seq_len):
    n_seq = len(feats) // seq_len
    return feats[:n_seq*seq_len].reshape(n_seq, seq_len, -1)

def make_seq_labels(labels, seq_len):
    n_seq = len(labels) // seq_len
    lbl = labels[:n_seq*seq_len].reshape(n_seq, seq_len)
    return (lbl.mean(axis=1) > 0.2).astype(int)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [16]:

class ChebConv(nn.Module):
    def __init__(self, in_dim, out_dim, K=3):
        super().__init__()
        self.K = K
        self.weights = nn.ParameterList([nn.Parameter(torch.randn(in_dim, out_dim)*0.1) for _ in range(K)])
    def forward(self, X, L):
        Tx = [X, L @ X]
        for k in range(2, self.K):
            Tx.append(2 * (L @ Tx[-1]) - Tx[-2])
        return sum(Tx[k] @ self.weights[k] for k in range(self.K))

class DeepChebGNN(nn.Module):
    """Two ChebConv layers with a nonlinearity between - more expressive than one linear conv."""
    def __init__(self, in_dim, hidden_dim=16, K=3):
        super().__init__()
        self.conv1 = ChebConv(in_dim, hidden_dim, K)
        self.conv2 = ChebConv(hidden_dim, in_dim, K)
        self.act = nn.ReLU()
    def forward(self, X, L):
        h = self.act(self.conv1(X, L))
        return self.conv2(h, L)
def build_channel_graph(chan_list):
    """Nodes = channels, edges = channels sharing the same letter-prefix subsystem (e.g. E-1, E-2, E-3)."""
    n = len(chan_list)
    adj = np.zeros((n, n))
    for i, ci in enumerate(chan_list):
        for j, cj in enumerate(chan_list):
            if i != j and ci.split("-")[0] == cj.split("-")[0]:
                adj[i, j] = 1.0
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(deg, 1e-8)))
    L = np.eye(n) - deg_inv_sqrt @ adj @ deg_inv_sqrt - np.eye(n)
    return torch.tensor(L, dtype=torch.float32)

In [17]:
channel_data = {}
skipped = []

for chan in np.unique(test_chan_ids):
    mask_train = train_chan_ids == chan
    Xc_train = X_train_raw[mask_train]
    mask_test = test_chan_ids == chan
    Xc_test = X_test[mask_test]
    yc_test = y_test[mask_test]

    if len(Xc_train) < SEQ_LEN*5 or len(Xc_test) < SEQ_LEN:
        skipped.append(chan)
        continue

    scaler = StandardScaler()
    scaler.fit(Xc_train)
    train_seq = make_sequences(Xc_train, SEQ_LEN)
    test_seq = make_sequences(Xc_test, SEQ_LEN)
    test_seq_labels = make_seq_labels(yc_test, SEQ_LEN)

    def scale_seq(seq):
        n, s, dd = seq.shape
        return scaler.transform(seq.reshape(-1, dd)).astype("float32").reshape(n, s, dd)
    train_seq_s, test_seq_s = scale_seq(train_seq), scale_seq(test_seq)

    n_val = max(1, int(0.2 * len(train_seq_s)))
    fit_seq, val_seq = train_seq_s[:-n_val], train_seq_s[-n_val:]
    if len(fit_seq) == 0 or len(val_seq) == 0:
        skipped.append(chan)
        continue

    fit_t = torch.tensor(fit_seq).to(device)
    val_t = torch.tensor(val_seq).to(device)
    test_t = torch.tensor(test_seq_s).to(device)

    model = LSTMAutoencoder(Xc_train.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    for _ in range(80):
        model.train()
        optimizer.zero_grad()
        out, _ = model(fit_t)
        loss = criterion(out, fit_t)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_out, val_z = model(val_t)
        val_err = ((val_out - val_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
        test_out, test_z = model(test_t)
        test_err = ((test_out - test_t) ** 2).mean(dim=(1, 2)).cpu().numpy()

    channel_data[chan] = {
        "val_err_mean": val_err.mean(), "val_err_std": val_err.std() + 1e-8,
        "node_feature": val_z.mean(dim=0).cpu().numpy(),
        "test_err": test_err, "test_z_latent": test_z.cpu().numpy(),
        "test_labels": test_seq_labels,
    }
    print(f"{chan:8s} | val_err_mean={val_err.mean():.4f} | n_train_seq={len(train_seq):3d} n_test_seq={len(test_seq):3d}")

print(f"\nSkipped {len(skipped)} channels: {skipped}")
print("Trained:", len(channel_data), "channels")

A-1      | val_err_mean=0.0000 | n_train_seq= 17 n_test_seq= 53
A-2      | val_err_mean=1.5797 | n_train_seq= 16 n_test_seq= 48
A-3      | val_err_mean=0.7692 | n_train_seq= 16 n_test_seq= 50
A-4      | val_err_mean=0.6955 | n_train_seq= 16 n_test_seq= 50
A-7      | val_err_mean=1.1642 | n_train_seq= 17 n_test_seq= 53
B-1      | val_err_mean=0.0000 | n_train_seq= 14 n_test_seq= 49
C-1      | val_err_mean=0.3725 | n_train_seq= 12 n_test_seq= 13
D-1      | val_err_mean=1.5833 | n_train_seq= 17 n_test_seq= 52
D-11     | val_err_mean=0.0500 | n_train_seq= 15 n_test_seq= 45
D-13     | val_err_mean=0.0000 | n_train_seq=  8 n_test_seq= 47
D-14     | val_err_mean=0.0000 | n_train_seq= 22 n_test_seq= 15
D-15     | val_err_mean=0.6224 | n_train_seq= 12 n_test_seq= 12
D-16     | val_err_mean=0.0310 | n_train_seq=  8 n_test_seq= 13
D-2      | val_err_mean=0.0000 | n_train_seq= 17 n_test_seq= 53
D-3      | val_err_mean=0.0161 | n_train_seq= 17 n_test_seq= 53
D-4      | val_err_mean=0.0210 | n_train

In [19]:
chan_list = list(channel_data.keys())
L_t = build_channel_graph(chan_list)

node_feat_dim = channel_data[chan_list[0]]["node_feature"].shape[0]
node_features = torch.tensor(
    np.stack([channel_data[c]["node_feature"] for c in chan_list]), dtype=torch.float32
).to(device)
L_t = L_t.to(device)

gnn = DeepChebGNN(node_feat_dim, hidden_dim=16, K=3).to(device)
opt_g = torch.optim.Adam(gnn.parameters(), lr=1e-2)
crit_g = nn.MSELoss()

for _ in range(200):
    gnn.train()
    opt_g.zero_grad()
    refined = gnn(node_features, L_t)
    loss = crit_g(refined, node_features)   # self-supervised: reconstruct via neighbor structure
    loss.backward()
    opt_g.step()

gnn.eval()
with torch.no_grad():
    refined_features = gnn(node_features, L_t).cpu().numpy()

# ---- combine LSTM error z-score + graph-distance z-score into final score ----
from sklearn.linear_model import LogisticRegression

all_tune_X, all_tune_y, all_final_X, all_final_y = [], [], [], []

for i, chan in enumerate(chan_list):
    cd = channel_data[chan]
    lstm_z = (cd["test_err"] - cd["val_err_mean"]) / cd["val_err_std"]
    graph_dist = np.linalg.norm(cd["test_z_latent"] - refined_features[i], axis=1)
    graph_z = (graph_dist - graph_dist.mean()) / (graph_dist.std() + 1e-8)

    y = cd["test_labels"]
    n = len(y)
    perm = np.random.permutation(n)
    n_tune = max(1, int(0.4 * n))
    tune_idx, final_idx = perm[:n_tune], perm[n_tune:]

    all_tune_X.append(np.column_stack([lstm_z[tune_idx], graph_z[tune_idx]]))
    all_tune_y.append(y[tune_idx])
    all_final_X.append(np.column_stack([lstm_z[final_idx], graph_z[final_idx]]))
    all_final_y.append(y[final_idx])

X_tune = np.concatenate(all_tune_X); y_tune = np.concatenate(all_tune_y)
X_final = np.concatenate(all_final_X); y_final = np.concatenate(all_final_y)

# learn the best way to combine lstm_z and graph_z, instead of guessing 50/50
combiner = LogisticRegression(class_weight="balanced")
combiner.fit(X_tune, y_tune)
prob_tune = combiner.predict_proba(X_tune)[:, 1]
prob_final = combiner.predict_proba(X_final)[:, 1]

candidates = np.percentile(prob_tune, np.arange(50, 100, 1))
best_thr, best_f1_tune = None, -1
for thr in candidates:
    f1 = f1_score(y_tune, (prob_tune > thr).astype(int), zero_division=0)
    if f1 > best_f1_tune:
        best_f1_tune, best_thr = f1, thr

final_preds = (prob_final > best_thr).astype(int)
print("Learned combiner weights (lstm_z, graph_z):", combiner.coef_)
print("Pooled threshold (tune F1):", best_f1_tune)
print("\nFinal Classification Report (TDA + LSTM + Deep GNN + learned combiner):")
print(classification_report(y_final, final_preds, zero_division=0))
print("Overall F1:", f1_score(y_final, final_preds, zero_division=0))

Learned combiner weights (lstm_z, graph_z): [[1.67533835e-09 2.74094780e-15]]
Pooled threshold (tune F1): 0.4336734693877551

Final Classification Report (TDA + LSTM + Deep GNN + learned combiner):
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      1535
           1       0.38      0.54      0.45       247

    accuracy                           0.82      1782
   macro avg       0.65      0.70      0.67      1782
weighted avg       0.85      0.82      0.83      1782

Overall F1: 0.4489112227805695


In [12]:
# channel_data = {}
# skipped = []

# for chan in np.unique(test_chan_ids):
#     mask_train = train_chan_ids == chan
#     Xc_train = X_train_raw[mask_train]
#     mask_test = test_chan_ids == chan
#     Xc_test = X_test[mask_test]
#     yc_test = y_test[mask_test]

#     if len(Xc_train) < SEQ_LEN*5 or len(Xc_test) < SEQ_LEN:
#         skipped.append(chan)
#         continue

#     scaler = StandardScaler()
#     scaler.fit(Xc_train)
#     train_seq = make_sequences(Xc_train, SEQ_LEN)
#     test_seq = make_sequences(Xc_test, SEQ_LEN)
#     test_seq_labels = make_seq_labels(yc_test, SEQ_LEN)

#     def scale_seq(seq):
#         n, s, dd = seq.shape
#         return scaler.transform(seq.reshape(-1, dd)).astype("float32").reshape(n, s, dd)
#     train_seq_s, test_seq_s = scale_seq(train_seq), scale_seq(test_seq)

#     n_val = max(1, int(0.2 * len(train_seq_s)))
#     fit_seq, val_seq = train_seq_s[:-n_val], train_seq_s[-n_val:]
#     if len(fit_seq) == 0 or len(val_seq) == 0:
#         skipped.append(chan)
#         continue

#     fit_t = torch.tensor(fit_seq).to(device)
#     val_t = torch.tensor(val_seq).to(device)
#     test_t = torch.tensor(test_seq_s).to(device)

#     model = LSTMAutoencoder(Xc_train.shape[1]).to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#     criterion = nn.MSELoss()

#     for epoch in range(80):
#         model.train()
#         optimizer.zero_grad()
#         out, _ = model(fit_t)
#         train_loss = criterion(out, fit_t)
#         train_loss.backward()
#         optimizer.step()

#     model.eval()
#     with torch.no_grad():
#         val_out, val_z = model(val_t)
#         val_loss = criterion(val_out, val_t).item()
#         val_err = ((val_out - val_t) ** 2).mean(dim=(1, 2)).cpu().numpy()
#         test_out, test_z = model(test_t)
#         test_err = ((test_out - test_t) ** 2).mean(dim=(1, 2)).cpu().numpy()

#     channel_data[chan] = {
#         "val_err_mean": val_err.mean(), "val_err_std": val_err.std() + 1e-8,
#         "node_feature": val_z.mean(dim=0).cpu().numpy(),
#         "test_err": test_err, "test_z_latent": test_z.cpu().numpy(),
#         "test_labels": test_seq_labels,
#     }
#     print(f"{chan:8s} | final_train_loss={train_loss.item():.4f} val_loss={val_loss:.4f} | "
#           f"n_train_seq={len(train_seq):3d} n_test_seq={len(test_seq):3d} anomalies={test_seq_labels.sum():2d}")

# print(f"\nSkipped {len(skipped)} channels: {skipped}")
# print("Trained:", len(channel_data), "channels")